# Experiment 4 — Full k×k Covariance CMA-ES vs Sep-CMA-ES

**Motivation**: Stage 1 used sep-CMA-ES (diagonal D_sub) because at n=3072 a full  
covariance matrix is unestimable. At k=40 it has only 1600 entries — fully tractable.  
The question: do cross-direction correlations in the k=40 subspace exist and are they  
worth learning?

## What the full covariance can capture

- **DCT–DCT correlations**: adjacent frequency components that the model responds  
  to together (e.g. both fy=1,fx=0 and fy=0,fx=1 matter for a horizontal edge).
- **DCT–superpixel correlations**: a frequency component that targets a specific  
  superpixel region (the frequency modulates the spatial patch).
- **Superpixel–superpixel correlations**: adjacent spatial regions that move together  
  (e.g. all patches belonging to the same object).

## Ablation design

| Variant | Covariance | λ | Queries/gen |
|---|---|---|---|
| sep-CMA-ES (baseline) | Diagonal D ∈ ℝ^k | 10 | 10 |
| full-CMA-ES λ=10 | Full C ∈ ℝ^(k×k) | 10 | 10 |
| full-CMA-ES λ=k | Full C ∈ ℝ^(k×k) | k=40 | 40 |
| sep-CMA-ES λ=k | Diagonal | k=40 | 40 |

All at k=40, budget 1000 queries.

In [ ]:
import sys, os
from pathlib import Path

_candidates = [Path.cwd(), Path.cwd()/'STAGE_2', Path.cwd()/'Adversial ML'/'STAGE_2']
for _p in _candidates:
    if (_p / 'utils_stage2.py').exists():
        sys.path.insert(0, str(_p.resolve())); break
else:
    raise FileNotFoundError('utils_stage2.py not found')

import numpy as np
import torch
import matplotlib.pyplot as plt

from utils_stage2 import (
    clip01, compute_ssim, compute_l2, compute_linf,
    Oracle, load_models, get_jointly_correct,
    phase1, phase2, build_subspace_with_dc, sep_cmaes,
    SSIM_STOP, H, W, C, N_QUICK, RANDOM_SEED,
)

np.random.seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

K_DCT, K_SP = 20, 20
P3_BUDGET   = 1000
os.makedirs('outputs/exp4', exist_ok=True)

In [ ]:
model_std, model_rob = load_models(device)
images = get_jointly_correct(model_std, model_rob, device, n=N_QUICK)
B = build_subspace_with_dc(k_dct=K_DCT, k_sp=K_SP)   # DC-inclusive
k = B.shape[0]
print(f'Images: {len(images)}   k={k}')

## Full k×k CMA-ES implementation

Standard CMA-ES with full covariance matrix.  
Eigendecomposition is performed once per generation (O(k³) = O(40³) = 64k ops, cheap).

In [ ]:
def full_cmaes(oracle, img_hwc, x_adv_init, B, lam, max_queries,
                ssim_stop=SSIM_STOP):
    """
    Full k×k CMA-ES in subspace B. No L-inf gate.
    Covariance named 'Cov' (not 'C') to avoid shadowing the C=3 channel constant.
    Returns (best_x, best_l2, best_ssim, queries_used, history).
    history entries include 'cov_trace' and 'cov_condition' diagnostics.
    """
    k  = B.shape[0]
    x0 = img_hwc.flatten()
    delta0  = x_adv_init.flatten() - x0
    theta_m = (B @ delta0).astype(np.float64)

    mu    = max(1, lam // 2)
    w_raw = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1, dtype=np.float64))
    w     = w_raw / w_raw.sum()
    mueff = 1.0 / np.sum(w**2)

    cc    = (4 + mueff/k) / (k + 4 + 2*mueff/k)
    cs    = (mueff + 2) / (k + mueff + 5)
    c1    = 2.0 / ((k + 1.3)**2 + mueff)
    cmu   = min(1 - c1, 2*(mueff - 2 + 1/mueff) / ((k + 2)**2 + mueff))
    damps = 1 + 2*max(0.0, np.sqrt((mueff - 1)/(k + 1)) - 1) + cs
    chiN  = k**0.5 * (1 - 1/(4*k) + 1/(21*k**2))

    Cov  = np.eye(k, dtype=np.float64)   # full covariance (Cov, not C)
    pc   = np.zeros(k, dtype=np.float64)
    ps   = np.zeros(k, dtype=np.float64)
    sigma = float(max(0.1 * np.linalg.norm(delta0) / np.sqrt(k), 1e-5))

    best_x    = x_adv_init.copy()
    best_l2   = compute_l2(img_hwc, best_x)
    best_ssim = compute_ssim(img_hwc, best_x)

    history, gen, queries = [], 0, 0

    while queries < max_queries:
        batch = min(lam, max_queries - queries)
        if batch <= 0:
            break

        evals, Q = np.linalg.eigh(Cov)
        evals = np.maximum(evals, 1e-20)
        sqrt_evals = np.sqrt(evals)

        zs = np.random.randn(batch, k)
        ys = (Q * sqrt_evals[None, :]) @ zs.T
        ys = ys.T                                   # (batch, k) samples from N(0,Cov)
        thetas = theta_m[None, :] + sigma * ys

        pool = []
        for i in range(batch):
            x_cand = clip01(x0 + B.T @ thetas[i].astype(np.float32)).reshape(H, W, C)
            is_adv, _ = oracle.query(x_cand, phase='p3')
            queries += 1
            if is_adv:                              # L-inf gate removed
                l2 = compute_l2(img_hwc, x_cand)
                pool.append((l2, zs[i], ys[i], thetas[i], x_cand))
                if l2 < best_l2:
                    best_l2, best_x = l2, x_cand.copy()
                    best_ssim = compute_ssim(img_hwc, best_x)

        cov_trace = float(np.trace(Cov))
        cov_cond  = float(evals.max() / (evals.min() + 1e-20))
        history.append({'gen': gen, 'queries': queries,
                        'best_l2': best_l2, 'best_ssim': best_ssim,
                        'sigma': sigma, 'cov_trace': cov_trace,
                        'cov_condition': cov_cond})

        if best_ssim >= ssim_stop:
            break

        if not pool:
            sigma = min(sigma * 2.0, 10.0)
            gen += 1
            continue

        pool.sort(key=lambda t: t[0])
        sel   = pool[:mu]
        n_sel = len(sel)
        w_s   = w[:n_sel] / w[:n_sel].sum()

        ys_sel = np.array([t[2] for t in sel])
        zs_sel = np.array([t[1] for t in sel])
        yw = (w_s[:, None] * ys_sel).sum(0)
        zw = (w_s[:, None] * zs_sel).sum(0)

        theta_m = theta_m + sigma * yw
        ps = (1 - cs)*ps + np.sqrt(cs*(2-cs)*mueff) * zw
        gen += 1
        hsig = (np.linalg.norm(ps)/chiN/np.sqrt(1-(1-cs)**(2*gen))) < (1.4+2/(k+1))
        pc = (1 - cc)*pc + hsig*np.sqrt(cc*(2-cc)*mueff) * yw

        rank_mu = sum(w_s[i] * np.outer(ys_sel[i], ys_sel[i]) for i in range(n_sel))
        Cov = ((1 - c1 - cmu) * Cov
               + c1 * (np.outer(pc, pc) + (1 - hsig)*cc*(2-cc)*Cov)
               + cmu * rank_mu)
        Cov = 0.5 * (Cov + Cov.T)

        sigma = float(np.clip(
            sigma * np.exp((cs/damps) * (np.linalg.norm(ps)/chiN - 1)),
            1e-10, 10.0))

    return best_x, best_l2, best_ssim, queries, history

## Run four-variant ablation

In [ ]:
MODELS = [('standard', model_std), ('robust', model_rob)]

VARIANTS = {
    'sep  λ=10':   lambda o, img, x0, B_: sep_cmaes(  o, img, x0, B_, lam=10,  max_queries=P3_BUDGET),
    'sep  λ=k':    lambda o, img, x0, B_: sep_cmaes(  o, img, x0, B_, lam=k,   max_queries=P3_BUDGET),
    'full λ=10':   lambda o, img, x0, B_: full_cmaes( o, img, x0, B_, lam=10,  max_queries=P3_BUDGET),
    'full λ=k':    lambda o, img, x0, B_: full_cmaes( o, img, x0, B_, lam=k,   max_queries=P3_BUDGET),
}

all_results  = {v: {m: [] for m, _ in MODELS} for v in VARIANTS}
all_histories = {v: {m: [] for m, _ in MODELS} for v in VARIANTS}

# Precompute Phase 1+2
p12 = {m: {} for m, _ in MODELS}
for model_name, model in MODELS:
    for rec in images:
        o = Oracle(model, rec['label'], device)
        xb, _ = phase1(o, rec['img'], seed=rec['idx'])
        if xb is None:
            p12[model_name][rec['idx']] = None
        else:
            xb = phase2(o, rec['img'], xb)
            p12[model_name][rec['idx']] = xb

print('Phase 1+2 done.')

In [ ]:
for model_name, model in MODELS:
    print(f'\n=== {model_name.upper()} ===')
    for vname, vfn in VARIANTS.items():
        for rec in images:
            xb = p12[model_name].get(rec['idx'])
            if xb is None:
                all_results[vname][model_name].append(None)
                all_histories[vname][model_name].append([])
                continue
            o = Oracle(model, rec['label'], device)
            out = vfn(o, rec['img'], xb, B)
            l2_p2 = compute_l2(rec['img'], xb)
            all_results[vname][model_name].append(
                dict(l2_p2=l2_p2, l2_p3=out[1], ssim=out[2],
                     improvement=l2_p2-out[1], q=out[3]))
            all_histories[vname][model_name].append(out[4])

        valid = [r for r in all_results[vname][model_name] if r]
        print(f'  {vname:<14}: med_L2={np.median([r["l2_p3"] for r in valid]):.4f}  '
              f'med_imp={np.median([r["improvement"] for r in valid]):.4f}')

## Plot 1 — L2 curves: sep vs full at both λ values

In [ ]:
COLORS = {'sep  λ=10': '#4C72B0', 'sep  λ=k': '#4C72B0',
          'full λ=10': '#DD8452', 'full λ=k': '#DD8452'}
STYLES = {'sep  λ=10': '-', 'sep  λ=k': '--',
          'full λ=10': '-', 'full λ=k': '--'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (model_name, _) in zip(axes, MODELS):
    for vname in VARIANTS:
        hists = [h for h in all_histories[vname][model_name] if h]
        if not hists:
            continue
        q_max  = max(h[-1]['queries'] for h in hists)
        q_grid = np.arange(1, q_max + 1)
        curves = []
        for hist in hists:
            qs  = np.array([h['queries']  for h in hist])
            l2s = np.array([h['best_l2'] for h in hist])
            curves.append(np.interp(q_grid, qs, l2s))
        arr = np.array(curves)
        med = np.median(arr, axis=0)
        ax.plot(q_grid, med, label=vname,
                color=COLORS[vname], ls=STYLES[vname])
        p25 = np.percentile(arr, 25, axis=0)
        p75 = np.percentile(arr, 75, axis=0)
        ax.fill_between(q_grid, p25, p75, alpha=0.08, color=COLORS[vname])

    ax.set_xlabel('Phase-3 queries')
    ax.set_ylabel('Best L2 (median)')
    ax.set_title(f'{model_name}')
    ax.legend()

plt.suptitle('Experiment 4 — Full vs sep CMA-ES, k=40', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/exp4/l2_curves.png', dpi=150)
plt.show()

## Plot 2 — Covariance condition number over time (full-CMA-ES diagnostic)

A high condition number means the covariance has learned a strongly preferred direction.  
If it grows throughout the run, the covariance is genuinely useful (not stale).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (model_name, _) in zip(axes, MODELS):
    for vname in ['full λ=10', 'full λ=k']:
        hists = [h for h in all_histories[vname][model_name] if h]
        if not hists:
            continue
        q_max  = max(h[-1]['queries'] for h in hists)
        q_grid = np.arange(1, q_max + 1)
        cond_curves = []
        for hist in hists:
            qs    = np.array([h['queries']        for h in hist])
            conds = np.array([h['cov_condition']  for h in hist])
            cond_curves.append(np.interp(q_grid, qs, conds))
        arr = np.array(cond_curves)
        ax.plot(q_grid, np.median(arr, axis=0), label=vname)

    ax.set_xlabel('Phase-3 queries')
    ax.set_ylabel('Covariance condition number (log scale)')
    ax.set_yscale('log')
    ax.set_title(f'{model_name}')
    ax.legend()

plt.suptitle('Experiment 4 — Covariance condition number evolution', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/exp4/cov_condition.png', dpi=150)
plt.show()

## Plot 3 — Learned covariance structure (top eigenvectors)

Visualise what directions the full-CMA-ES learned to prefer,  
by plotting the top eigenvectors of the final covariance matrix  
projected back to pixel space.

In [ ]:
def full_cmaes_with_final_Cov(oracle, img_hwc, x_adv_init, B, lam, max_queries):
    """Same as full_cmaes but also returns the final covariance matrix."""
    k  = B.shape[0]
    x0 = img_hwc.flatten()
    delta0  = x_adv_init.flatten() - x0
    theta_m = (B @ delta0).astype(np.float64)

    mu    = max(1, lam // 2)
    w_raw = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1, dtype=np.float64))
    w     = w_raw / w_raw.sum()
    mueff = 1.0 / np.sum(w**2)
    cc    = (4 + mueff/k) / (k + 4 + 2*mueff/k)
    cs    = (mueff + 2) / (k + mueff + 5)
    c1    = 2.0 / ((k + 1.3)**2 + mueff)
    cmu   = min(1 - c1, 2*(mueff - 2 + 1/mueff) / ((k + 2)**2 + mueff))
    damps = 1 + 2*max(0.0, np.sqrt((mueff - 1)/(k + 1)) - 1) + cs
    chiN  = k**0.5 * (1 - 1/(4*k) + 1/(21*k**2))

    Cov = np.eye(k, dtype=np.float64)   # renamed from C to avoid shadowing C=3
    pc, ps = np.zeros(k), np.zeros(k)
    sigma = float(max(0.1 * np.linalg.norm(delta0) / np.sqrt(k), 1e-5))
    best_x  = x_adv_init.copy()
    best_l2 = compute_l2(img_hwc, best_x)
    gen, queries = 0, 0

    while queries < max_queries:
        batch = min(lam, max_queries - queries)
        if batch <= 0:
            break
        evals, Q = np.linalg.eigh(Cov)
        evals = np.maximum(evals, 1e-20)
        zs = np.random.randn(batch, k)
        ys = (Q * np.sqrt(evals)[None, :]) @ zs.T; ys = ys.T
        thetas = theta_m[None, :] + sigma * ys
        pool = []
        for i in range(batch):
            x_cand = clip01(x0 + B.T @ thetas[i].astype(np.float32)).reshape(H, W, C)
            is_adv, _ = oracle.query(x_cand, phase='p3')
            queries += 1
            if is_adv:                              # L-inf gate removed
                l2 = compute_l2(img_hwc, x_cand)
                pool.append((l2, zs[i], ys[i], x_cand))
                if l2 < best_l2:
                    best_l2, best_x = l2, x_cand.copy()
        if not pool:
            sigma = min(sigma * 2.0, 10.0); gen += 1; continue
        pool.sort(key=lambda t: t[0])
        sel = pool[:mu]; n_sel = len(sel); w_s = w[:n_sel] / w[:n_sel].sum()
        ys_sel = np.array([t[2] for t in sel])
        zs_sel = np.array([t[1] for t in sel])
        yw = (w_s[:, None] * ys_sel).sum(0)
        zw = (w_s[:, None] * zs_sel).sum(0)
        theta_m = theta_m + sigma * yw
        ps = (1-cs)*ps + np.sqrt(cs*(2-cs)*mueff)*zw
        gen += 1
        hsig = (np.linalg.norm(ps)/chiN/np.sqrt(1-(1-cs)**(2*gen))) < (1.4+2/(k+1))
        pc = (1-cc)*pc + hsig*np.sqrt(cc*(2-cc)*mueff)*yw
        rank_mu = sum(w_s[i]*np.outer(ys_sel[i],ys_sel[i]) for i in range(n_sel))
        Cov = (1-c1-cmu)*Cov + c1*(np.outer(pc,pc)+(1-hsig)*cc*(2-cc)*Cov) + cmu*rank_mu
        Cov = 0.5*(Cov+Cov.T)
        sigma = float(np.clip(sigma*np.exp((cs/damps)*(np.linalg.norm(ps)/chiN-1)),1e-10,10.0))

    return best_x, Cov   # return best image and final covariance


rec       = next(r for r in images if p12['standard'].get(r['idx']) is not None)
xb        = p12['standard'][rec['idx']]
o         = Oracle(model_std, rec['label'], device)
_, Cov_final = full_cmaes_with_final_Cov(o, rec['img'], xb, B, lam=10, max_queries=500)

evals, evecs = np.linalg.eigh(Cov_final)
evals = evals[::-1]
evecs = evecs[:, ::-1]

n_show = 6
fig, axes = plt.subplots(1, n_show, figsize=(14, 3))
for i in range(n_show):
    v_pixel = B.T @ evecs[:, i].astype(np.float32)
    v_img   = v_pixel.reshape(H, W, C)         # C=3 here — safe, no name conflict
    vmax    = np.abs(v_img).max() + 1e-8
    display = 0.5 + 0.5 * v_img / vmax
    axes[i].imshow(np.clip(display, 0, 1))
    axes[i].set_title(f'ev {i+1}\nλ={evals[i]:.3f}', fontsize=8)
    axes[i].axis('off')

plt.suptitle('Experiment 4 — Top eigenvectors of learned Cov (pixel space)', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/exp4/cov_eigenvectors.png', dpi=150)
plt.show()
print('Experiment 4 complete. Outputs saved to outputs/exp4/')